# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [51]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


We have one row per content for each date on the month of march

In [52]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")

relevant = [f for f in files if "fact_content_daily_performance" in f]
for f in relevant:
    print(f)

fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month=2026-04/data_0.parquet
fact_content_daily_performance/month=202

In [53]:
from datasets import load_dataset
cols_needed = [ 'content_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks',  'gsc_avg_position', 'gsc_data_available', 'ga4_data_available', 'scroll_events']
ds_march = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/data_0.parquet",
    split="train",
    token=hf_token
)

df_march = ds_march.select_columns(cols_needed).to_pandas()
print(df_march.shape)
print(df_march['report_date'].min(), df_march['report_date'].max())

df_march_slim = df_march[['report_date', 'content_hash_id']]

daily_check = df_march_slim.groupby('report_date')['content_hash_id'].agg(
    total_rows='count',
    unique_ids='nunique'
).reset_index()

mismatches = daily_check[daily_check['total_rows'] != daily_check['unique_ids']]
print(f"Dates with duplicate content_hash_id: {len(mismatches)}")

print("Therefore we have one row per content(page), for each day on the month of March")



(9841378, 8)
2026-03-01 2026-03-31
Dates with duplicate content_hash_id: 0
Therefore we have one row per content(page), for each day on the month of March


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**How I'm picking features**for each one I ask:
1. Visibility: does it describe (or help derive) how visible the page is?
2. Performance: does it describe (or help derive) how well the page engages people?
3. Trend direction: does it describe (or help derive) which way things are moving over time?
4. Market opportunity: does it describe (or help derive) how hard the page is to rank for?
5. Content investment:  is it an attribute of the content itself that feeds into performance/visibility?

I don't need a feature for every single one of these just picking the strongest 5 overall.

**My 5 features:**
- **gsc_impressions** (visibility)
- **gsc_avg_position**  (visibility)
- **ctr** (derived from gsc_clicks / gsc_impressions) performance
- **trend_direction** (derived from report_date, comparing early vs late month) trend
- **word_count** (content investment)

**Label:**
No real labelthis is unsupervised. The archetypes are names I assign to
clusters after the fact, based on where they sit across visibility/performance/
trend. Not something pulled from a column.

**Constraints:**
**report_date** isn't a feature on its own, but I use it to derive **trend_direction**
by comparing early-month vs late-month values within March.

**Excluded (and why):**
- **content_hash_id, client_hash_id, keyword_hash_id, url_hash_id** these
  are hashed for privacy, so they carry no real meaning as features. I only use
  them for joins/grouping, never as signal.
- GA4 columns **ga4_*,sessions_*, `ai_*, scroll_events**  GA4 availability
  is only ~4.2% of March, way too sparse to build anything reliable on. **scroll_events**
  gets cut for the same reason since it's a GA4-tracked interaction and won't fire
  without GA4 data present.
- **search_volume,competition,competition_level, cpc** these are real
  market-opportunity signals and I considered using them, but I'm capped at 5
  features and visibility/performance/trend/content-investment felt like the
  core of what defines an archetype. Cutting market-opportunity for scope, not
  because it's irrelevant.
- **keyword_token_count**close call against **word_count**, but token count is
  more about the keyword being targeted (search intent) than the content itself.
  **word_count** is a more direct signal of the actual investment put into the page.
- **gsc_click** kept as raw signal underneath **ctr**, but not counted separately
  since it's already folded into the derived metric.




## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [54]:

from huggingface_hub import HfApi

api = HfApi(token=hf_token)
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")

dim_files = [f for f in files if "dim_content" in f]
for f in dim_files:
    print(f)

dim_content.parquet


In [55]:

cols_needed = [ 'content_hash_id', 'word_count', 'is_published', 'content_updated_date']
ds_content = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="dim_content.parquet",
    split="train",
    token=hf_token
)

df_content = ds_content.select_columns(cols_needed).to_pandas()


In [56]:
df_march_joined = df_march.merge(
    df_content,
    on='content_hash_id',
    how='left',
    validate='m:1'
)

In [57]:

print("joined data size", df_march.shape[0], "un-joined data size", df_march_joined.shape[0])
print("same size we can continue")
print("filter using TRUE for either of them and both")
df_ga4_gsc_march = df_march_joined[(df_march_joined['gsc_data_available'] == True)& (df_march_joined['ga4_data_available'] == True)]
df_ga4_march = df_march_joined[(df_march_joined['ga4_data_available'] == True)]
df_gsc_march = df_march_joined[df_march_joined['gsc_data_available'] == True]
print("both ga4 and gsc availability",(df_ga4_gsc_march.shape[0]/ df_march_joined.shape[0]) * 100)
print("ga4 availability",(df_ga4_march.shape[0]/ df_march_joined.shape[0]) * 100)
print("gsc avialability", (df_gsc_march.shape[0]/ df_march_joined.shape[0]) * 100)

gsc_available = df_march[df_march['gsc_data_available'] == True]
print(f"Rows where gsc_data_available IS TRUE: {gsc_available.shape[0]:,} out of {df_march.shape[0]:,} ({gsc_available.shape[0]/df_march.shape[0]:.1%})")


df_march_joined  = df_march_joined[df_march_joined['gsc_data_available'] == True]

joined data size 9841378 un-joined data size 9841378
same size we can continue
filter using TRUE for either of them and both
both ga4 and gsc availability 3.7021949568444583
ga4 availability 4.206382480177065
gsc avialability 36.69263592964319
Rows where gsc_data_available IS TRUE: 3,611,061 out of 9,841,378 (36.7%)


In [58]:
##Granularity
import gc
df_march_grain = df_march_joined[['report_date', 'content_hash_id']]

grain = df_march_grain.groupby('report_date')['content_hash_id'].agg(
    total_rows='count',
    unique_ids='nunique'
).reset_index()
print(grain)
print("Data has low granularity, Content is aggregated daily")

del grain
del df_march_grain

gc.collect()

   report_date  total_rows  unique_ids
0   2026-03-01      101910      101910
1   2026-03-02      103696      103696
2   2026-03-03      107362      107362
3   2026-03-04      109377      109377
4   2026-03-05      109740      109740
5   2026-03-06      110037      110037
6   2026-03-07      102153      102153
7   2026-03-08      101170      101170
8   2026-03-09      111313      111313
9   2026-03-10      113051      113051
10  2026-03-11      113329      113329
11  2026-03-12      114224      114224
12  2026-03-13      114066      114066
13  2026-03-14      113497      113497
14  2026-03-15      115312      115312
15  2026-03-16      119966      119966
16  2026-03-17      121714      121714
17  2026-03-18      122200      122200
18  2026-03-19      121281      121281
19  2026-03-20      120548      120548
20  2026-03-21      120472      120472
21  2026-03-22      122193      122193
22  2026-03-23      122515      122515
23  2026-03-24      124920      124920
24  2026-03-25      12606

940

In [59]:
print(df_march['scroll_events'].isna().sum() / len(df_march))
print((df_march['scroll_events'] == 0).sum() / len(df_march))

0.30673966592889734
0.6807580198626656


In [60]:
#Count
print(f"Total rows in March 2026 partition: {df_march.shape[0]:,}")
print(f"Date range: {df_march['report_date'].min()} to {df_march['report_date'].max()}")
print(f"Number of distinct days: {df_march['report_date'].nunique()}")

Total rows in March 2026 partition: 9,841,378
Date range: 2026-03-01 to 2026-03-31
Number of distinct days: 31


In [61]:
#The trap
""""
Intentionally including statistcs from a future reference i.e Word count, and using cluster labels to train.
"""
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df_march_joined = df_march_joined[[ 'content_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'gsc_data_available', 'is_published', 'word_count']]
print(df_march_joined.shape)
print(df_march_joined.info)



(3611061, 8)
<bound method DataFrame.info of                   content_hash_id report_date  gsc_impressions  gsc_clicks  \
0        content_b7e512995f79d5a6  2026-03-01               20           0   
1        content_05597932fe4da067  2026-03-01                1           0   
2        content_7a105f548d9c6916  2026-03-01              125           1   
3        content_905aa32a0230694e  2026-03-01                7           0   
4        content_a3ea9792f793ec72  2026-03-01               11           0   
...                           ...         ...              ...         ...   
9841373  content_f712a9db831acfd5  2026-03-31              154           1   
9841374  content_1b80768a4d22c0af  2026-03-31               27           0   
9841375  content_a4e0a726b424d6e6  2026-03-31               63           0   
9841376  content_5a8a45c79112787c  2026-03-31              381           0   
9841377  content_66097d8adf63762f  2026-03-31              263           2   

         gsc_avg_p

In [62]:
key_cols = ['content_hash_id', 'report_date']


dupe_mask = df_march_joined.duplicated(subset=key_cols, keep=False)
dupes = df_march_joined[dupe_mask].sort_values(key_cols)
print(f"{dupe_mask.sum()} rows share content_hash_id + report_date")
dupes.head(20)


0 rows share content_hash_id + report_date


,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_data_available,is_published,word_count


In [63]:
df_march_joined = df_march_joined[df_march_joined['is_published'] == True]

df_march_joined['CTR'] = (df_march_joined['gsc_clicks'] / df_march_joined['gsc_impressions']) * 100

df_march_joined.dtypes

,0
content_hash_id,object
report_date,object
gsc_impressions,int64
gsc_clicks,int64
gsc_avg_position,float64
gsc_data_available,bool
is_published,bool
word_count,float64
CTR,float64


In [64]:
import pandas as pd
df_march_joined.sort_values(by=['report_date', 'content_hash_id'], ascending=True, inplace=True)
df_march_joined['report_date'] = pd.to_datetime(df_march_joined['report_date'])
pos_diff = df_march_joined.groupby('content_hash_id')['gsc_avg_position'].diff()
days_diff = df_march_joined.groupby('content_hash_id')['report_date'].diff().dt.days
df_march_joined['trend_direction'] = (pos_diff/days_diff).fillna(0)

In [65]:
df_march_joined.head()

,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_data_available,is_published,word_count,CTR,trend_direction
634735,content_0000d495bfbfb4a8,2026-03-01,9,0,1.333333,True,True,2832.0,0.0,0.0
126100,content_00014efc121d911d,2026-03-01,2,0,6.000000,True,True,NaN,0.0,0.0
236031,content_000184dde41afe75,2026-03-01,188,0,3.670213,True,True,2587.0,0.0,0.0
108180,content_0002bd310bf01f15,2026-03-01,1,0,72.000000,True,True,NaN,0.0,0.0
732747,content_00032be2df0005ca,2026-03-01,22,0,10.545455,True,True,1502.0,0.0,0.0


In [66]:
df_march_joined.describe()

,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,word_count,CTR,trend_direction
count,3609108,3.609108e+06,3.609108e+06,3.609108e+06,2.405369e+06,3.609108e+06,3.609108e+06
mean,2026-03-16 13:21:34.752277504,7.775287e+01,2.276676e-01,1.583121e+01,2.938650e+03,3.080704e-01,6.140185e-02
min,2026-03-01 00:00:00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-4.880000e+02
25%,2026-03-09 00:00:00,4.000000e+00,0.000000e+00,3.741935e+00,2.454000e+03,0.000000e+00,-2.360606e+00
50%,2026-03-17 00:00:00,1.600000e+01,0.000000e+00,7.500000e+00,2.769000e+03,0.000000e+00,0.000000e+00
75%,2026-03-24 00:00:00,6.200000e+01,0.000000e+00,2.021841e+01,3.246000e+03,0.000000e+00,2.444444e+00
max,2026-03-31 00:00:00,4.008400e+04,2.740000e+02,4.980000e+02,2.934100e+04,1.000000e+02,4.920000e+02
std,NaN,2.499345e+02,1.277555e+00,1.985972e+01,1.132651e+03,3.008250e+00,1.341368e+01


In [67]:
missing_ctr = df_march_joined[df_march_joined["CTR"].isna()]
missing_ctr[["gsc_impressions", "gsc_avg_position", "is_published"]].describe()

,gsc_impressions,gsc_avg_position
count,0.0,0.0
mean,NaN,NaN
std,NaN,NaN
min,NaN,NaN
25%,NaN,NaN
50%,NaN,NaN
75%,NaN,NaN
max,NaN,NaN


In [68]:
df_march_joined['pos_bucket'] = pd.cut(df_march_joined['gsc_avg_position'], bins=[0,3,10,20,100,500])

In [69]:

df_march_joined['word_count'] = df_march_joined.groupby('pos_bucket', observed=True)['word_count'].transform(
    lambda x: x.fillna(x.median())
)

In [70]:
import numpy as np
for col in ["gsc_impressions", "gsc_avg_position", "word_count", "CTR"]:
    df_march_joined[col] = np.log1p(df_march_joined[col])

for col in ["gsc_impressions", "word_count", "CTR"]:
    median  = df_march_joined[col].median()
    df_march_joined[col] = df_march_joined[col].fillna(median)

In [71]:
df_march_joined["trend_direction"] = np.sign(df_march_joined["trend_direction"]) * np.log1p(np.abs(df_march_joined["trend_direction"]))

In [72]:
df_march_joined.head()

,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_data_available,is_published,word_count,CTR,trend_direction,pos_bucket
634735,content_0000d495bfbfb4a8,2026-03-01,2.302585,0,0.847298,True,True,7.949091,0.0,0.0,"(0, 3]"
126100,content_00014efc121d911d,2026-03-01,1.098612,0,1.945910,True,True,7.899895,0.0,0.0,"(3, 10]"
236031,content_000184dde41afe75,2026-03-01,5.241747,0,1.541205,True,True,7.858641,0.0,0.0,"(3, 10]"
108180,content_0002bd310bf01f15,2026-03-01,0.693147,0,4.290459,True,True,8.003864,0.0,0.0,"(20, 100]"
732747,content_00032be2df0005ca,2026-03-01,3.135494,0,2.446292,True,True,7.315218,0.0,0.0,"(10, 20]"


In [73]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

df_march_joined["CTR"] = df_march_joined["CTR"].fillna(0)
df_march_joined["trend_direction_log"] = np.sign(df_march_joined["trend_direction"]) * np.log1p(np.abs(df_march_joined["trend_direction"]))

feature_cols = ["gsc_impressions", "gsc_avg_position", "word_count", "CTR", "trend_direction"]



X = df_march_joined[feature_cols].dropna()


scaler = QuantileTransformer(output_distribution='normal', random_state=42)
X_scaled = scaler.fit_transform(X).astype(np.float32)
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_pca), size=min(1_000_000, len(X_pca)), replace=False)
X_sample = X_pca[sample_idx]

best_k = 3
#Leakage from word_count
km_honest = KMeans(n_clusters=best_k, n_init=10, random_state=42, max_iter=300)
labels_honest = km_honest.fit_predict(X_sample)
score_honest = silhouette_score(X_sample, labels_honest, sample_size=5_000, random_state=42)
print(f"honest silhouette (k={best_k}): {score_honest:.5f}")

df_sample = pd.DataFrame(X_sample, columns=['PC1', 'PC2', 'PC3'])
df_sample["cluster_id_LEAKED"] = labels_honest
X_leaky = df_sample.values
#Leakage from cluster labels
km_leaky = KMeans(n_clusters=best_k, n_init=10, random_state=42, max_iter=300)
labels_leaky = km_leaky.fit_predict(X_leaky)
score_leaky = silhouette_score(X_leaky, labels_leaky, sample_size=5_000, random_state=42)
print(f"leaky silhouette (cluster_id_LEAKED included as a feature): {score_leaky:.4f}")

jump = score_leaky - score_honest
relative = (score_leaky / score_honest - 1) * 100
print(f"jump from leakage: {jump:.4f} ({relative:.1f}% relative increase)")

df_sample = df_sample.drop(columns=["cluster_id_LEAKED"])
print(f"leaked column removed — honest score kept as the real result: {score_honest:.5f}")

honest silhouette (k=3): 0.68923
leaky silhouette (cluster_id_LEAKED included as a feature): 0.6945
jump from leakage: 0.0053 (0.8% relative increase)
leaked column removed — honest score kept as the real result: 0.68923


Leakage from:
1. Cluster labels
2. Word_count containing data from the future, that clusters will be used to evaluate

## 4. Data limits

What this data can't tell you, based on what I actually checked:

**GA4 is basically unusable for this lane.** Only 4.2% of March rows have GA4
data available. That means for over 95% of content, I have zero idea what
happened once someone actually landed on the page, no pageviews, no sessions,
no engagement time, nothing. So any archetype that depends on "how well does
this page engage people once they arrive" just isn't answerable here. I'm
stuck inferring engagement indirectly through CTR, which is a search behavior
proxy, not a real on-site engagement measure.

**GSC coverage is only about 37%, and the missing 63% isn't random.** Content
without GSC data could mean a few different things, too new to have ranking
data yet, not indexed by Google at all, or just below whatever threshold
Search Console uses to report data. The data has no way to tell these
situations apart. So my archetypes only describe the ~37% slice that actually
has measurable search behavior. I can't say anything meaningful about the
other 63% of content, they're just invisible to this analysis, not "bad."

**The panel is growing, not fixed.** When I checked row counts per day, March 1
had about 276k rows and March 31 had about 331k. That's new content getting
added to the tracked set throughout the month, not the same set of pages
showing up every day. So when I compare early-March to late-March for trend,
some of that "difference" is really just newer content that wasn't around yet
on day one, not actual performance change.

**One month isn't enough to call something a real trend.** My trend_direction
feature is a slope calculated across March's 31 days, which is better than
just comparing two snapshots, but it's still only a month. A page could look
like it's "rising" just from normal week to week noise, or because of the
panel growth thing above, not because it's actually on a real upward trajectory.
To trust a trend label, I'd really want to see it hold up across two or three
months, not one.

**Nothing here tells you why.** Even if I find a clean group of pages that all
share a pattern, like low visibility but high CTR, the data can't tell me why
that's happening. Could be content quality, could be who's competing for that
keyword, could be a seasonal blip. Clustering shows me the pattern, not the
cause.

If I had to pick just one limitation to call out as the big one, it's probably
the growing panel issue, since it directly messes with the trend feature I'm
building, and I can actually point to the row counts as proof.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.